# Model A – Standard CNN (EuroSAT, 10 land-use classes)

EN3150 Assignment 03 – custom CNN that interleaves **standard 2D convolutions** and **max-pooling**.

**Data:** the shared EuroSAT split built by `Dataset.ipynb` in `data/eurosat/train|val|test/<class>/`
(64×64 RGB satellite images, 18,900 / 4,050 / 4,050). Run `Dataset.ipynb` once before this notebook.
No data augmentation.

In [1]:
import os
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models

DATA_DIR    = Path("data/eurosat")   # built by Dataset.ipynb
RESULTS_DIR = Path("results")        # numbers and saved model (format: results/README.md)
FIGURES_DIR = Path("figures")        # figures for the report
RESULTS_DIR.mkdir(exist_ok=True)
FIGURES_DIR.mkdir(exist_ok=True)

IMG_SIZE = (64, 64)
SEED     = 42
BATCH    = 32
AUTOTUNE = tf.data.AUTOTUNE

tf.keras.utils.set_random_seed(SEED)
print("TensorFlow", tf.__version__, "| GPU:", tf.config.list_physical_devices("GPU") or "none")

TensorFlow 2.21.0 | GPU: none


In [2]:
def load_split(name, shuffle):
    return tf.keras.utils.image_dataset_from_directory(
        str(DATA_DIR / name), image_size=IMG_SIZE, batch_size=BATCH,
        shuffle=shuffle, seed=SEED, label_mode="int")

train_raw = load_split("train", shuffle=True)
val_raw   = load_split("val",   shuffle=False)
test_raw  = load_split("test",  shuffle=False)

CLASS_NAMES = train_raw.class_names
NUM_CLASSES = len(CLASS_NAMES)
print(NUM_CLASSES, "classes:", CLASS_NAMES)

# Scale pixels from 0-255 to 0-1. Keep the decoded images in memory (cache) and
# prepare the next batch while the current one trains (prefetch).
def scale(x, y):
    return tf.cast(x, tf.float32) / 255.0, y

train_ds = train_raw.map(scale, num_parallel_calls=AUTOTUNE).cache().prefetch(AUTOTUNE)
val_ds   = val_raw.map(scale,   num_parallel_calls=AUTOTUNE).cache().prefetch(AUTOTUNE)
test_ds  = test_raw.map(scale,  num_parallel_calls=AUTOTUNE).cache().prefetch(AUTOTUNE)

Found 18900 files belonging to 10 classes.


Found 4050 files belonging to 10 classes.


Found 4050 files belonging to 10 classes.


10 classes: ['AnnualCrop', 'Forest', 'HerbaceousVegetation', 'Highway', 'Industrial', 'Pasture', 'PermanentCrop', 'Residential', 'River', 'SeaLake']


## 1. Architecture

Four blocks of **Conv2D (3×3) → BatchNorm → ReLU → MaxPooling (2×2)** with filters doubling
32 → 64 → 128 → 256, then **Global Average Pooling → Dense(128, ReLU) → Dropout(0.5) → Dense(10, softmax)**.

| Layer | Output shape | Kernel / units | Parameters |
|---|---|---|---|
| Input | 64×64×3 | – | 0 |
| Conv2D (no bias) | 64×64×32 | 3×3, 32 filters | 3·3·3·32 = 864 |
| BatchNorm + ReLU | 64×64×32 | – | 4·32 = 128 (64 trainable) |
| MaxPool | 32×32×32 | 2×2 | 0 |
| Conv2D (no bias) | 32×32×64 | 3×3, 64 filters | 3·3·32·64 = 18,432 |
| BatchNorm + ReLU | 32×32×64 | – | 4·64 = 256 (128 trainable) |
| MaxPool | 16×16×64 | 2×2 | 0 |
| Conv2D (no bias) | 16×16×128 | 3×3, 128 filters | 3·3·64·128 = 73,728 |
| BatchNorm + ReLU | 16×16×128 | – | 4·128 = 512 (256 trainable) |
| MaxPool | 8×8×128 | 2×2 | 0 |
| Conv2D (no bias) | 8×8×256 | 3×3, 256 filters | 3·3·128·256 = 294,912 |
| BatchNorm + ReLU | 8×8×256 | – | 4·256 = 1,024 (512 trainable) |
| MaxPool | 4×4×256 | 2×2 | 0 |
| GlobalAvgPool | 256 | – | 0 |
| Dense + ReLU | 128 | 128 units | 256·128 + 128 = 32,896 |
| Dropout | 128 | rate 0.5 | 0 |
| Dense + Softmax | 10 | 10 units | 128·10 + 10 = 1,290 |
| **Total** | | | **424,042** (423,082 trainable + 960 BatchNorm statistics) |

Conv params = K·K·C_in·C_out (no bias – BatchNorm adds its own shift). BatchNorm has 4 values per channel:
γ and β are trained, the running mean and variance are statistics. Dense params = inputs·units + units.

Batch normalisation was added after comparing variants on validation data: it raised validation accuracy by
about 4 percentage points at the same size, and it costs nothing at inference because it can be folded
into the preceding convolution.

In [3]:
def build_model_a(input_shape=IMG_SIZE + (3,), num_classes=NUM_CLASSES):
    """Model A – standard CNN. Calling it again gives a model with fresh weights."""
    inputs = layers.Input(shape=input_shape)
    x = inputs
    for filters in [32, 64, 128, 256]:
        x = layers.Conv2D(filters, kernel_size=3, padding="same", use_bias=False)(x)
        x = layers.BatchNormalization()(x)
        x = layers.ReLU()(x)
        x = layers.MaxPooling2D(pool_size=2)(x)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(128, activation="relu")(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(num_classes, activation="softmax")(x)
    return models.Model(inputs, outputs, name="Model_A_StandardCNN")

model_a = build_model_a()
model_a.summary()

Model: "Model_A_StandardCNN"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 64, 64, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 64, 64, 32)     │           864 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 64, 64, 32)     │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu (ReLU)                    │ (None, 64, 64, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 32, 32, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 32, 32, 64)     │        18,432 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 32, 32, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu_1 (ReLU)                  │ (None, 32, 32, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 16, 16, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 16, 16, 128)    │        73,728 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 16, 16, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu_2 (ReLU)                  │ (None, 16, 16, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 8, 8, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 8, 8, 256)      │       294,912 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 8, 8, 256)      │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu_3 (ReLU)                  │ (None, 8, 8, 256)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 4, 4, 256)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 256)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 10)             │         1,290 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 424,042 (1.62 MB)

 Trainable params: 423,082 (1.61 MB)

 Non-trainable params: 960 (3.75 KB)

In [4]:
# Check: a batch of 64x64x3 images gives one probability vector of length 10 per image
x_batch, y_batch = next(iter(val_ds))
probs = model_a.predict(x_batch, verbose=0)
n_trainable = int(sum(np.prod(w.shape) for w in model_a.trainable_weights))
print("pixel range       :", float(x_batch.numpy().min()), "to", float(x_batch.numpy().max()))
print("input  shape      :", x_batch.shape)
print("output shape      :", probs.shape)
print("rows sum to 1     :", bool(np.allclose(probs.sum(axis=1), 1.0, atol=1e-5)))
print("total params      :", f"{model_a.count_params():,}")
print("trainable params  :", f"{n_trainable:,}")
assert probs.shape == (len(x_batch), NUM_CLASSES)

pixel range       : 0.09803921729326248 to 1.0
input  shape      : (32, 64, 64, 3)
output shape      : (32, 10)
rows sum to 1     : True
total params      : 424,042
trainable params  : 423,082


## 2. Parameters and MACs per layer

Parameters are calculated **by formula** from each layer's kernel size and channel counts, then checked
against Keras. MACs (multiply-accumulate operations) are counted for **one 64×64 image** – the work an edge
device does per prediction.

| Layer | Parameters | MACs per image |
|---|---|---|
| Conv k×k, C_in → C_out, no bias, 'same' padding | k·k·C_in·C_out | H_out·W_out·k·k·C_in·C_out |
| BatchNorm, C channels | 4C (2C trainable γ, β + 2C running mean / variance) | 0 at inference – folded into the conv |
| Dense n_in → n_out | n_in·n_out + n_out | n_in·n_out |
| MaxPool, ReLU, GAP, Dropout | 0 | ≈ 0 (comparisons / additions only) |

In [5]:
def layer_table(model):
    """One row per layer that has weights; parameters from the formulas above."""
    rows = []
    for layer in model.layers:
        if isinstance(layer, layers.Conv2D):
            h, w, c_out = layer.output.shape[1:]
            c_in = layer.input.shape[-1]
            k = layer.kernel_size[0]
            p = k * k * c_in * c_out + (c_out if layer.use_bias else 0)
            rows.append(dict(layer=f"Conv2D {c_in}->{c_out}", kernel=f"{k}x{k}", filters_units=c_out,
                             output=f"{h}x{w}x{c_out}", params=p, trainable=p, macs=h * w * k * k * c_in * c_out))
        elif isinstance(layer, layers.BatchNormalization):
            h, w, c = layer.output.shape[1:]
            rows.append(dict(layer=f"BatchNorm ({c})", kernel="", filters_units=c, output=f"{h}x{w}x{c}",
                             params=4 * c, trainable=2 * c, macs=0))
        elif isinstance(layer, layers.Dense):
            n_in, n_out = layer.input.shape[-1], layer.units
            p = n_in * n_out + n_out
            rows.append(dict(layer=f"Dense {n_in}->{n_out}", kernel="", filters_units=n_out,
                             output=str(n_out), params=p, trainable=p, macs=n_in * n_out))
    return pd.DataFrame(rows)

table = layer_table(model_a)
totals = table[["params", "trainable", "macs"]].sum()
keras_trainable = int(sum(np.prod(w.shape) for w in model_a.trainable_weights))

fmt = {c: "{:,}".format for c in ["params", "trainable", "macs"]}
print(table.to_string(index=False, formatters=fmt))
print()
print(f"Total parameters     : {totals.params:>11,}   Keras: {model_a.count_params():,}")
print(f"Trainable parameters : {totals.trainable:>11,}   Keras: {keras_trainable:,}")
print(f"MACs per 64x64 image : {totals.macs:>11,}   (~{totals.macs / 1e6:.1f} million)")
assert totals.params == model_a.count_params() and totals.trainable == keras_trainable, "formula != Keras"
print("Check passed: hand-calculated totals match Keras")

MODEL_A_MACS = int(totals.macs)
table.to_csv(RESULTS_DIR / "model_a_layers.csv", index=False)

          layer kernel  filters_units    output  params trainable       macs
   Conv2D 3->32    3x3             32  64x64x32     864       864  3,538,944
 BatchNorm (32)                    32  64x64x32     128        64          0
  Conv2D 32->64    3x3             64  32x32x64  18,432    18,432 18,874,368
 BatchNorm (64)                    64  32x32x64     256       128          0
 Conv2D 64->128    3x3            128 16x16x128  73,728    73,728 18,874,368
BatchNorm (128)                   128 16x16x128     512       256          0
Conv2D 128->256    3x3            256   8x8x256 294,912   294,912 18,874,368
BatchNorm (256)                   256   8x8x256   1,024       512          0
 Dense 256->128                   128       128  32,896    32,896     32,768
  Dense 128->10                    10        10   1,290     1,290      1,280

Total parameters     :     424,042   Keras: 424,042
Trainable parameters :     423,082   Keras: 423,082
MACs per 64x64 image :  60,196,096   (~60.2 mill

In [6]:
# Where do the parameters and the computation sit? (per block)
names = ["Block 1 (32)", "Block 2 (64)", "Block 3 (128)", "Block 4 (256)", "Head (Dense)"]
per_block = pd.DataFrame(0, index=names, columns=["trainable", "macs"])
block = -1
for _, r in table.iterrows():
    if r.layer.startswith("Conv2D"):
        block += 1
    per_block.loc[names[block] if r.layer.startswith(("Conv2D", "BatchNorm")) else names[-1]] += [r.trainable, r.macs]

fig, ax = plt.subplots(1, 2, figsize=(12, 3.6))
ax[0].bar(names, per_block.trainable / 1e3, color="#1f77b4"); ax[0].set_title("Trainable parameters (thousands)")
ax[1].bar(names, per_block.macs / 1e6, color="#ff7f0e");    ax[1].set_title("MACs per image (millions)")
for a in ax:
    a.tick_params(axis="x", rotation=20); a.grid(axis="y", alpha=0.3)
plt.tight_layout(); plt.savefig(FIGURES_DIR / "model_a_cost_per_block.png", dpi=150); plt.show()

per_block["share of params"] = (per_block.trainable / per_block.trainable.sum()).map("{:.1%}".format)
per_block["share of MACs"] = (per_block.macs / per_block.macs.sum()).map("{:.1%}".format)
per_block

,trainable,macs,share of params,share of MACs
Block 1 (32),928,3538944,0.2%,5.9%
Block 2 (64),18560,18874368,4.4%,31.4%
Block 3 (128),73984,18874368,17.5%,31.4%
Block 4 (256),295424,18874368,69.8%,31.4%
Head (Dense),34186,34048,8.1%,0.1%


## 3. Design justification

**Kernel size 3×3.** The smallest kernel that still looks at a neighbourhood. Stacking 3×3 layers with
pooling in between grows the receptive field cheaply – after the four blocks one output value sees the
whole 64×64 patch. A 5×5 kernel would need 25 / 9 ≈ 2.8× the weights and MACs in every layer.

**Filters 32 → 64 → 128 → 256.** Each 2×2 max-pool keeps only ¼ of the positions, so doubling the filters
keeps the work of blocks 2–4 equal (18.9 M MACs each, see the chart) while deeper layers get more channels
for more abstract features: colours and edges → textures (crops, roofs, water) → field shapes, road lines →
whole land-use pattern. Block 1 runs at full resolution, so starting with only 32 filters keeps it cheap
(3.5 M MACs). Block 4 holds 70 % of the parameters but costs no more than blocks 2 and 3 because its
feature maps are only 8×8.

**Global average pooling + Dense 128 + Dropout 0.5.** GAP turns the 4×4×256 feature map into 256 values,
so the hidden dense layer needs 256·128 weights instead of 4·4·256·128 = 524,288 with `Flatten` – a smaller
model and less overfitting. 128 units are enough to combine the features for 10 classes; dropout 0.5
regularises this layer. The output layer has one unit per class (10).

**Batch normalisation.** Normalises each channel during training, so training is faster and more stable
(it raised validation accuracy by about 4 points in our comparison). It costs **nothing at inference**:
its scale and shift are folded into the weights of the preceding convolution before deployment, which is
also why the convolutions need no bias.

**Hardware-aware activation choice – ReLU.**
- ReLU is `max(0, x)`: one comparison per value, no multiplication, no exponential. Sigmoid and tanh need
  `exp()`, which on a microcontroller without a floating-point unit costs tens to hundreds of cycles or a
  lookup table in scarce memory.
- It works directly on int8 values after quantisation (a clamp at the zero point), and inference engines
  such as TensorFlow Lite fuse it into the preceding convolution, so it adds practically no cost.
- Its gradient is 0 or 1, so gradients do not vanish and training converges quickly.
- Many activations become exactly 0, which sparse / zero-skipping accelerators can exploit.

**Softmax only at the output.** Softmax uses `exp`, but it is applied once, to 10 values. For a prediction
the device only needs the arg-max of the logits, so softmax can be skipped entirely on the edge device.

## 4. Optimizer and learning-rate comparison

Every run starts from a **freshly built Model A with the same random seed** (identical initial weights), uses
the same data, batch size 128 and the same **20 epochs** with a constant learning rate (no early stopping, no
schedule), so differences come from the optimizer only. Choices are made on the **validation** set; the test
set is not used.

| Run | Optimizer | Learning rate | Momentum | Purpose |
|---|---|---|---|---|
| 1–3 | Adam | 0.01, 0.003, 0.001 | – | choose the learning rate of the chosen optimizer |
| 4 | SGD (plain) | 0.05 | 0 | baseline asked for in the brief |
| 5–7 | SGD + Momentum | 0.05 | 0.5, 0.9, 0.99 | effect of the momentum parameter |

In [7]:
class EpochTimer(tf.keras.callbacks.Callback):
    """Records the wall-clock time of every epoch."""
    def on_train_begin(self, logs=None): self.times = []
    def on_epoch_begin(self, epoch, logs=None): self._t = time.time()
    def on_epoch_end(self, epoch, logs=None): self.times.append(time.time() - self._t)

def make_optimizer(name, lr, momentum=0.0):
    if name == "adam":
        return tf.keras.optimizers.Adam(learning_rate=lr)
    if name == "sgd":
        return tf.keras.optimizers.SGD(learning_rate=lr, momentum=momentum)
    raise ValueError(f"unknown optimizer {name!r}")

def train_batches(batch):
    """Training data in batches of `batch`, re-shuffled every epoch (val / test keep their order)."""
    return (train_ds.unbatch().shuffle(18_900, seed=SEED, reshuffle_each_iteration=True)
            .batch(batch).prefetch(AUTOTUNE))

In [8]:
SWEEP_EPOCHS = 20
SWEEP_BATCH  = 128
RUNS = [  # (label, optimizer, learning rate, momentum)
    ("Adam lr=0.01",        "adam", 1e-2, 0.0),
    ("Adam lr=0.003",       "adam", 3e-3, 0.0),
    ("Adam lr=0.001",       "adam", 1e-3, 0.0),
    ("SGD lr=0.05",         "sgd",  5e-2, 0.0),
    ("SGD+M 0.5 lr=0.05",   "sgd",  5e-2, 0.5),
    ("SGD+M 0.9 lr=0.05",   "sgd",  5e-2, 0.9),
    ("SGD+M 0.99 lr=0.05",  "sgd",  5e-2, 0.99),
]

# The runs take a while, so their histories are saved; re-running the notebook re-uses them.
# Set RERUN_SWEEP = True to train all seven runs again.
RERUN_SWEEP = False
SWEEP_FILE = RESULTS_DIR / "model_a_optimizers.json"

if SWEEP_FILE.exists() and not RERUN_SWEEP:
    sweep = json.loads(SWEEP_FILE.read_text())
    print(f"Loaded saved runs from {SWEEP_FILE} (set RERUN_SWEEP = True to retrain)\n")
else:
    sweep, sweep_train = {}, train_batches(SWEEP_BATCH)
    for label, opt, lr, mom in RUNS:
        tf.keras.backend.clear_session()
        tf.keras.utils.set_random_seed(SEED)                 # identical initial weights for every run
        m = build_model_a()
        m.compile(optimizer=make_optimizer(opt, lr, mom),
                  loss="sparse_categorical_crossentropy", metrics=["accuracy"])
        timer = EpochTimer()
        h = m.fit(sweep_train, validation_data=val_ds, epochs=SWEEP_EPOCHS, callbacks=[timer], verbose=0).history
        sweep[label] = dict(optimizer=opt, lr=lr, momentum=mom,
                            history={k: [float(v) for v in vals] for k, vals in h.items()},
                            epoch_times=timer.times)
        SWEEP_FILE.write_text(json.dumps(sweep))             # saved after every run
        print(f"finished {label}")

for label, r in sweep.items():
    h = r["history"]
    print(f"{label:<20} best val acc {max(h['val_accuracy']):.4f} (epoch {int(np.argmax(h['val_accuracy'])) + 1:>2})"
          f" | final val loss {h['val_loss'][-1]:.3f} | {np.mean(r['epoch_times'][1:]):.0f} s/epoch")

Loaded saved runs from results/model_a_optimizers.json (set RERUN_SWEEP = True to retrain)

Adam lr=0.01         best val acc 0.8407 (epoch 17) | final val loss 2.006 | 54 s/epoch
Adam lr=0.003        best val acc 0.9015 (epoch 15) | final val loss 0.433 | 58 s/epoch
Adam lr=0.001        best val acc 0.8928 (epoch 13) | final val loss 0.636 | 56 s/epoch
SGD lr=0.05          best val acc 0.8227 (epoch  8) | final val loss 1.934 | 58 s/epoch
SGD+M 0.5 lr=0.05    best val acc 0.8395 (epoch 16) | final val loss 8.665 | 58 s/epoch
SGD+M 0.9 lr=0.05    best val acc 0.8662 (epoch 12) | final val loss 0.825 | 52 s/epoch
SGD+M 0.99 lr=0.05   best val acc 0.5114 (epoch 11) | final val loss 2.289 | 57 s/epoch


In [9]:
def first_epoch_reaching(values, threshold):
    hits = [i + 1 for i, v in enumerate(values) if v >= threshold]
    return hits[0] if hits else None

opt_table = pd.DataFrame([{
    "run": label,
    "best val acc": max(r["history"]["val_accuracy"]),
    "epoch of best": int(np.argmax(r["history"]["val_accuracy"])) + 1,
    "final val acc": r["history"]["val_accuracy"][-1],
    "final val loss": r["history"]["val_loss"][-1],
    "first epoch val acc >= 90%": first_epoch_reaching(r["history"]["val_accuracy"], 0.90),
    "std val acc, last 5 epochs": float(np.std(r["history"]["val_accuracy"][-5:])),
} for label, r in sweep.items()])
opt_table.to_csv(RESULTS_DIR / "model_a_optimizers.csv", index=False)
opt_table.round(4)

,run,best val acc,epoch of best,final val acc,final val loss,first epoch val acc >= 90%,"std val acc, last 5 epochs"
0,Adam lr=0.01,0.8407,17,0.6516,2.0060,NaN,0.0693
1,Adam lr=0.003,0.9015,15,0.8854,0.4331,15.0,0.0917
2,Adam lr=0.001,0.8928,13,0.8388,0.6361,NaN,0.0654
3,SGD lr=0.05,0.8227,8,0.5580,1.9338,NaN,0.0751
4,SGD+M 0.5 lr=0.05,0.8395,16,0.2632,8.6645,NaN,0.2138
5,SGD+M 0.9 lr=0.05,0.8662,12,0.7704,0.8249,NaN,0.1135
6,SGD+M 0.99 lr=0.05,0.5114,11,0.2805,2.2887,NaN,0.0728


In [10]:
fig, ax = plt.subplots(2, 2, figsize=(14, 9))
groups = [("Adam – learning rate", [k for k in sweep if k.startswith("Adam")]),
          ("SGD – effect of momentum (lr = 0.05)", [k for k in sweep if k.startswith("SGD")])]
for col, (title, keys) in enumerate(groups):
    for k in keys:
        h = sweep[k]["history"]; ep = range(1, len(h["loss"]) + 1)
        line, = ax[0, col].plot(ep, h["val_loss"], label=k)
        ax[0, col].plot(ep, h["loss"], ls=":", color=line.get_color(), alpha=0.7)
        ax[1, col].plot(ep, h["val_accuracy"], label=k, color=line.get_color())
    ax[0, col].set_title(f"{title}\nloss (solid = validation, dotted = training)")
    ax[1, col].set_title(f"{title}\nvalidation accuracy")
    ax[0, col].set_ylim(0, 2.5); ax[1, col].set_ylim(0, 1)
    for a in ax[:, col]:
        a.set_xlabel("epoch"); a.grid(alpha=0.3); a.legend()
plt.tight_layout(); plt.savefig(FIGURES_DIR / "model_a_optimizers.png", dpi=150); plt.show()

### Discussion – which optimizer and learning rate, and what momentum does

**Chosen: Adam with learning rate 0.003.** It reached the highest validation accuracy of all seven runs
(90.2 %, epoch 15), was the only run to pass 90 % within 20 epochs, and also ended highest (88.5 % at
epoch 20, final validation loss 0.43). Adam keeps a separate, adaptive step size for every weight (from running
averages of the gradient and its square), so it needs little tuning and converges quickly – useful when each
training run is expensive.

**Learning rate (Adam 0.01 / 0.003 / 0.001).**
- **0.01 is too large:** validation accuracy only reached 84.1 % and dropped to 65.2 % in the last epoch; the
  validation loss ends at 2.0 although the training loss keeps falling – the steps are too big to settle.
- **0.001 is close to 0.003** (best 89.3 %) and reaches the lowest training loss (0.15), but its validation accuracy
  swings more (83.9 % at epoch 20); with only 20 epochs it is slightly behind.
- **0.003** is the best compromise between speed and stability, so it is used for the final training.

**Plain SGD vs SGD with momentum (all lr = 0.05).** Momentum adds a running average of past gradients to every
step: `v ← β·v + g`, `w ← w − lr·v`. Along directions where the gradient keeps the same sign the steps add up
(effective step ≈ lr / (1 − β)), while zig-zagging components cancel.

| Momentum β | effective step lr/(1−β) | best val acc | training loss after 5 / 20 epochs | behaviour |
|---|---|---|---|---|
| 0 (plain SGD) | 0.05 | 82.3 % | 0.57 / 0.23 | slowest, best epoch already at 8 |
| 0.5 | 0.10 | 84.0 % | 0.50 / 0.19 | faster convergence – lowest training loss |
| 0.9 | 0.50 | **86.6 %** | 0.61 / 0.22 | best SGD accuracy, but larger swings |
| 0.99 | 5.0 | 51.1 % | 1.58 / 1.77 | **diverges** – training loss never decreases |

- **Convergence speed:** a moderate momentum (0.5) lowers the training loss faster than plain SGD, and 0.9 reaches
  the best SGD validation accuracy (+4.3 points over plain SGD).
- **Stability:** at a fixed learning rate a larger momentum also makes the steps larger, so validation accuracy
  oscillates more (std over the last 5 epochs: 0.075 plain, 0.11 for 0.9). At 0.99 the effective step is 100× the
  learning rate and training breaks down. Momentum therefore has to be tuned **together** with the learning rate –
  with β = 0.9, a smaller learning rate (e.g. 0.005) gives the same effective step as plain SGD at 0.05 and would be the setting to try next (not run here).
- Even the best SGD run stays below Adam at 0.003 / 0.001 within the same 20 epochs.

**Why the validation curves are so jumpy.** All runs use a *constant* learning rate and Model A has batch
normalisation: the running mean / variance used at validation time lag behind the fast-changing weights, so one
epoch can score 90 % and the next 70 % although the training loss falls smoothly. This is why the final training
(section 5) **halves the learning rate when the validation loss stops improving** and keeps the weights of the best
validation epoch.

In [11]:
# Optimizer and learning rate used for the final training of Model A
CHOSEN = dict(label="Adam lr=0.003", optimizer="adam", lr=3e-3, momentum=0.0)
print("Chosen:", CHOSEN)

Chosen: {'label': 'Adam lr=0.003', 'optimizer': 'adam', 'lr': 0.003, 'momentum': 0.0}
